<a href="https://colab.research.google.com/github/LINWOO0099/PANDAS-DF/blob/main/FAQ%20semantic%20search%20engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
pip install chromadb


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 903.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentele

In [11]:
from sentence_transformers import SentenceTransformer
import chromadb

# -----------------------------
# FAQ Knowledge Base
# -----------------------------
faq_data = [
    {
        "id": "doc1",
        "question": "How do I reset my password?",
        "answer": "Click 'Forgot Password' on the login page and follow the instructions.",
        "category": "account",
    },
    {
        "id": "doc2",
        "question": "How can I change my email address?",
        "answer": "Go to Settings > Account > Email and update your email.",
        "category": "account",
    },
    {
        "id": "doc3",
        "question": "How do I cancel my subscription?",
        "answer": "Go to Settings > Billing > Cancel Subscription and confirm.",
        "category": "billing",
    },
    {
        "id": "doc4",
        "question": "What payment methods are accepted?",
        "answer": "We accept Visa, Mastercard, and UPI.",
        "category": "billing",
    },
    {
        "id": "doc5",
        "question": "Where can I download the mobile app?",
        "answer": "Search for our app on the App Store or Google Play.",
        "category": "support",
    },
    {
        "id": "doc6",
        "question": "How do I contact customer support?",
        "answer": "Email us at support@example.com or use the in-app chat.",
        "category": "support",
    },
]

# -----------------------------
# Load Embedding Model
# -----------------------------
model = SentenceTransformer("all-MiniLM-L6-v2")

# -----------------------------
# Create ChromaDB Collection
# (indexed only once)
# -----------------------------
client = chromadb.Client()

collection = client.get_or_create_collection(
    name="faq_collection",
    metadata={"hnsw:space": "cosine"}
)

# Avoid duplicate indexing if collection already contains data
if collection.count() == 0:
    documents = [item["question"] for item in faq_data]
    embeddings = model.encode(documents).tolist()
    ids = [item["id"] for item in faq_data]

    metadatas = [
        {
            "category": item["category"],
            "answer": item["answer"],
        }
        for item in faq_data
    ]

    collection.add(
        ids=ids,
        documents=documents,
        embeddings=embeddings,
        metadatas=metadatas,
    )

# -----------------------------
# Search Function
# -----------------------------
def search_faq(query, category_filter=None):
    """
    Search FAQ knowledge base using semantic similarity.

    Args:
        query (str): User query.
        category_filter (str, optional): Category filter.

    Returns:
        str: Best matching answer or fallback message.
    """

    query_embedding = model.encode([query]).tolist()

    query_params = {
        "query_embeddings": query_embedding,
        "n_results": 1,
    }

    if category_filter:
        query_params["where"] = {"category": category_filter}

    results = collection.query(**query_params)

    if not results["ids"] or not results["ids"][0]:
        return "I'm sorry, I don't have information on this topic."

    distance = results["distances"][0][0]

    # Chroma cosine distance -> similarity
    similarity = 1 - distance

    if similarity < 0.4:
        return "I'm sorry, I don't have information on this topic."

    return results["metadatas"][0][0]["answer"]


# -----------------------------
# Test Cases
# -----------------------------
print("Test Case 1:")
print(search_faq("I forgot my login credentials"))

print("\nTest Case 2:")
print(search_faq("I want to stop paying", category_filter="billing"))


print("\nTest Case 3:")
print(search_faq("What is the best recipe for biryani?"))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Test Case 1:
Click 'Forgot Password' on the login page and follow the instructions.

Test Case 2:
Go to Settings > Billing > Cancel Subscription and confirm.

Test Case 3:
I'm sorry, I don't have information on this topic.
